# Домашняя работа №1. Решение

**ФИО:** Малиновский Максимилиан
**Поток:** пока не указан

Задания - в [hw01_task.ipynb](hw01_task.ipynb). Здесь только решение.

Перед сдачей: **Kernel → Restart Kernel and Run All Cells.** Ноутбук обязан пройти сверху донизу без ошибок.

---

## 0. Самопроверка окружения

Эта ячейка выполняется первой всегда. Если она падает - остальное чинить бессмысленно.

In [ ]:
import sys
from pathlib import Path

# чтобы работал импорт из src/, даже если ноутбук запущен из папки notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print("интерпретатор:", sys.executable)
print("корень проекта:", ROOT)

assert ".venv" in sys.executable, "Ядро не из .venv проекта - выберите ядро 'Python (ml-course)'"
print("OK: ядро из .venv проекта")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

for lib in (pd, np):
    print(lib.__name__, lib.__version__)

---

## Задание 1. Форк, клон, remote

Форк: https://github.com/victoryfanfare/ml_learning_course

Рабочая ветка: `hw01`.

Вывод `git remote -v`:

```text
origin  https://github.com/victoryfanfare/ml_learning_course.git (fetch)
origin  https://github.com/victoryfanfare/ml_learning_course.git (push)
upstream  https://github.com/PolukarovIvan/ml_learning_course.git (fetch)
upstream  https://github.com/PolukarovIvan/ml_learning_course.git (push)
```

`origin` — мой форк для отправки решений. `upstream` — репозиторий преподавателя для получения новых заданий.

Локальная копия сначала скачана из репозитория преподавателя с именем remote `upstream`, затем к ней добавлен мой форк как `origin`.

Добавление проверяющих: ещё не выполнено, нужны GitHub-ники от куратора.


---

## Задание 2. Терминал

Команды выполнены в Ubuntu (WSL2) из корня проекта:

```bash
pwd                 # /home/mechrevo/projects/ml_learning_course
ls -a               # все файлы, включая скрытые
cat .gitignore      # правила исключения файлов из git
ls notebooks/       # hw01_task.ipynb и hw01_solution.ipynb
```

**Абсолютный и относительный путь:** абсолютный начинается от корня файловой системы, например `/home/mechrevo/projects/ml_learning_course/data/file.csv`. Относительный, например `data/file.csv`, отсчитывается от текущей рабочей папки. Если запустить ноутбук из `notebooks/`, такой путь будет искать файл в `notebooks/data/`, поэтому файл может не найтись. В проекте полезно строить пути от найденного корня проекта.

**Почему `ls` не показывает `.env`:** в Linux имена, начинающиеся с точки, считаются скрытыми. Команда `ls -a` показывает их. Это соглашение об отображении, а не защита содержимого файла.


---

## Задание 3. Окружение

Установлена Ubuntu 26.04 в WSL2. Системный Python имеет другую версию, поэтому Python 3.11 установлен отдельно через `uv python install 3.11`. Затем из корня проекта выполнены:

```bash
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python -m ipykernel install --user --name ml-course --display-name "Python (ml-course)"
python -m pip check
```

Проверенный интерпретатор: `/home/mechrevo/projects/ml_learning_course/.venv/bin/python`, Python 3.11.18. Проверка зависимостей: `No broken requirements found.` Ядро называется `Python (ml-course)`.

`.venv` — отдельный набор Python-библиотек для проекта. Активация добавляет его программы в начало `PATH`; явный вызов `.venv/bin/python` тоже использует это окружение. `requirements.txt` позволяет восстановить прямые зависимости с теми же версиями на другой машине.


---

## Задание 4. Секреты

Выполнено `cp .env.example .env`.

Вывод `git check-ignore -v .env`:

```text
.gitignore:2:.env    .env
```

В `git status --short` файл `.env` отсутствует.

**Почему `.env.example` в git лежит, а `.env` — нет:** шаблон описывает необходимые переменные и содержит публичные настройки. Локальный `.env` может содержать личные пароли. Если добавить пароль в коммит, он останется в истории даже после удаления из следующей версии файла.


---

## Задание 5. Подключение к базе

In [ ]:
from src.db import get_client, query_df

client = get_client()
print(client.query("SELECT version()").result_rows)

**Откуда `get_client` берёт хост и пароль:** при импорте `src.db` функция `load_dotenv()` загружает переменные из `.env` в окружение процесса. Затем `get_client()` читает их через `os.getenv`. Декоратор `lru_cache` позволяет повторно использовать один клиент в текущем процессе.

**Что будет, если не создать `.env`:** если необходимые переменные уже заданы в окружении, подключение может работать и без файла. Если их нигде нет, `_required` выдаст понятную ошибку о незаданном `CLICKHOUSE_HOST` или `CLICKHOUSE_USER`. Пароль допускается пустой для публичного доступа.


---

## Задание 6. Знакомство с датасетом

### 6.1. Сколько всего поездок

In [ ]:
# База считает строки, Python получает только одно число.
trip_count_df = query_df("SELECT count() AS trips FROM nyc_taxi.trips_small")
display(trip_count_df)


### 6.2. За какой период данные

In [ ]:
period_df = query_df("""
    SELECT min(pickup_datetime) AS first_pickup,
           max(pickup_datetime) AS last_pickup
    FROM nyc_taxi.trips_small
""")
display(period_df)


### 6.3. Первые 10 строк

In [ ]:
# Без ORDER BY понятие «первые» не задаёт определённого порядка.
sample_df = query_df("""
    SELECT * FROM nyc_taxi.trips_small
    ORDER BY pickup_datetime, trip_id
    LIMIT 10
""")
display(sample_df)


---

## Задание 7. Своя аналитика

### Вопрос 1: отличаются ли длительность и средняя скорость поездок в будни и выходные?

Будни — понедельник–пятница, выходные — суббота и воскресенье, без отдельного учёта праздников. Длительность считаем по разнице времени высадки и посадки. Убираем неположительный чек, расстояние вне (0; 100] миль, длительность вне [1; 180] минут и скорость вне [1; 100] миль/ч. Это явные учебные допущения для ограничения аномалий, а не универсальное определение правильных данных.

`GROUP BY` собирает поездки в две группы; `avg` вычисляет среднее, `quantileExact(0.5)` — медиану, центральное значение распределения. Средняя скорость ниже — среднее скоростей отдельных поездок.


In [ ]:
week_sql = """
SELECT
    if(toDayOfWeek(pickup_datetime) <= 5, 'Будни', 'Выходные') AS day_type,
    count() AS trips,
    round(avg(trip_distance), 2) AS avg_distance_miles,
    round(avg(duration_min), 2) AS avg_duration_min,
    round(quantileExact(0.5)(duration_min), 2) AS median_duration_min,
    round(avg(trip_distance / duration_min * 60), 2) AS avg_speed_mph
FROM (
    SELECT *, dateDiff('second', pickup_datetime, dropoff_datetime) / 60.0 AS duration_min
    FROM nyc_taxi.trips_small
    WHERE trip_distance > 0 AND trip_distance <= 100 AND total_amount > 0
)
WHERE duration_min BETWEEN 1 AND 180
  AND trip_distance / duration_min * 60 BETWEEN 1 AND 100
GROUP BY day_type
ORDER BY day_type
"""
week_df = query_df(week_sql)
display(week_df)


**Вывод:** после фильтрации в будни осталось 2 103 370 поездок, в выходные — 862 581. В будни средняя длительность равна 14,41 минуты, в выходные — 13,01 минуты, то есть примерно на 9,7% меньше. При этом средняя скорость выросла с 11,80 до 13,54 миль/ч, примерно на 14,7%. Среднее расстояние в выходные даже немного больше: 3,20 против 3,05 мили. Результат согласуется с менее загруженными дорогами, но не доказывает эту причину: состав маршрутов и время поездок тоже могут отличаться. Сырые количества поездок здесь нельзя сравнивать как дневной спрос, поскольку будних и выходных дней разное число.


### Вопрос 2: какие направленные маршруты между разными районами встречаются чаще всего?

Район посадки и район высадки вместе определяют маршрут: обратное направление считается отдельно. Убираем пустые названия районов, поездки внутри одного района и записи с неположительными расстоянием, чеком или длительностью.

Запрос с `GROUP BY pickup_ntaname, dropoff_ntaname`, подсчётом `count()` и сортировкой по убыванию вынесен в функцию `top_routes` в `src/taxi.py` (задание 9). Параметр `n` задаёт число строк в результате. Код запроса доступен ниже для проверки; импортированная функция используется повторно.


In [ ]:
import inspect
from src.taxi import top_routes

print(inspect.getsource(top_routes))
routes_df = top_routes(10)
display(routes_df)


**Вывод:** самый частый маршрут — Midtown-Midtown South → Hudson Yards-Chelsea-Flatiron-Union Square: 55 307 поездок, средний чек 10,46 доллара. Обратное направление почти так же популярно: 54 929 поездок и 10,76 доллара. Разница в количестве составляет только 378 поездок. Среди лидеров заметны маршруты между районами центрального Манхэттена. Это рейтинг отфильтрованных межрайонных поездок, а не всех перемещений по Нью-Йорку: поездки внутри одного района намеренно исключены.


### Вопрос 3: как число пассажиров связано со средним чеком поездки?

Сравним группы от 1 до 6 пассажиров, одновременно посмотрев медианный чек и среднее расстояние. Исключаем неположительные расстояние и чек, расстояние больше 100 миль и длительность вне [1; 180] минут. Итоговая сумма относится ко всей поездке, а не к одному пассажиру.


In [ ]:
passenger_sql = """
SELECT
    passenger_count AS passengers,
    count() AS trips,
    round(avg(total_amount), 2) AS avg_total,
    round(quantileExact(0.5)(total_amount), 2) AS median_total,
    round(avg(trip_distance), 2) AS avg_distance_miles
FROM nyc_taxi.trips_small
WHERE passenger_count BETWEEN 1 AND 6
  AND trip_distance > 0 AND trip_distance <= 100
  AND total_amount > 0
  AND dateDiff('second', pickup_datetime, dropoff_datetime) BETWEEN 60 AND 10800
GROUP BY passenger_count
ORDER BY passengers
"""
passenger_df = query_df(passenger_sql)
display(passenger_df)
print(passenger_df.to_markdown(index=False, floatfmt=".2f"))


**Вывод:** средний чек для одного пассажира составляет 15,92 доллара; для двух — 17,07; трёх — 16,72; четырёх — 17,14; пяти — 16,37; шести — 16,01. Последовательного роста с числом пассажиров нет. Максимум — у четырёх пассажиров, на 1,22 доллара (около 7,7%) выше группы с одним пассажиром. Но у этой группы больше и среднее расстояние: 3,44 против 3,01 мили. Медианы чека находятся в узком диапазоне 11,80–12,35 доллара. Поэтому нельзя заключить, что пассажиры сами по себе увеличивают стоимость: различаются маршруты, расстояния и другие характеристики поездок.


---

## Задание 8. График

График иллюстрирует вывод из вопроса №1: в выходные отфильтрованные поездки в среднем короче по времени и быстрее. Две панели показывают разные величины с разными единицами измерения. Числа над столбцами позволяют проверить разницу без оценки на глаз.


In [ ]:
plot_df = week_df.set_index("day_type").loc[["Будни", "Выходные"]]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, column, title, ylabel in [
    (axes[0], "avg_duration_min", "Длительность поездки", "Средняя длительность, минуты"),
    (axes[1], "avg_speed_mph", "Скорость поездки", "Средняя скорость, мили/ч"),
]:
    bars = ax.bar(plot_df.index, plot_df[column], color=["#3569a8", "#df8b30"], width=0.55)
    ax.set_title(title)
    ax.set_xlabel("Тип дня")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, plot_df[column].max() * 1.2)
    ax.bar_label(bars, fmt="%.2f", padding=4)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
fig.suptitle("Такси Нью-Йорка: будни и выходные, июль–сентябрь 2015")
fig.tight_layout()
evidence_dir = ROOT / "data" / "hw01_evidence"
evidence_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(evidence_dir / "weekdays_weekends.png", dpi=160, bbox_inches="tight")
plt.show()


---

## Задание 9. Код в `src/` и зависимости

Функция `top_routes(n)` вынесена в `src/taxi.py` и уже использована в вопросе №2. Она получает данные через `query_df` из `src.db`. Аргумент проверяется и передаётся как типизированный SQL-параметр, а не вставляется в текст запроса.

Ниже — два вызова с разным числом маршрутов. `head()` здесь не заменяет параметр: ограничение числа возвращаемых строк выполняется самой базой через `LIMIT`.


In [ ]:
from src.taxi import top_routes

top_3 = top_routes(3)
top_5 = top_routes(5)
print("Три маршрута:")
display(top_3)
print("Пять маршрутов:")
display(top_5)
assert len(top_3) == 3 and len(top_5) == 5


**Какую библиотеку добавили и зачем:** `tabulate==0.9.0`, для читаемых текстовых таблиц через `DataFrame.to_markdown()`. Она используется при выводе результата вопроса №3. Библиотека установлена в `.venv`, версия зафиксирована в `requirements.txt`.


---

## Задание 10. Агент

**Чем `allow` отличается от `deny`:** в `.claude/settings.json` перечислены разрешённые и запрещённые действия агента Claude. `allow` разрешает подходящие команды, а `deny` запрещает их и имеет приоритет при совпадении правил. Это настройки конкретного инструмента: другой агент может применять собственные настройки доступа.

**Почему `git push`, `rm -rf` и чтение `.env` в `deny`:** push публикует изменения на сервере, рекурсивное удаление может уничтожить файлы, а чтение `.env` может раскрыть секреты. Эти ограничения уменьшают последствия ошибки агента.

**Что просили у агента, что он предложил, что пришлось исправить руками:** я попросил Codex пройти домашнюю работу с пояснениями для новичка. Агент скачал проект, настроил `origin` и `upstream`, создал ветку `hw01` и помог установить Ubuntu. Я создал форк в своём аккаунте и прошёл начальную настройку пользователя Ubuntu. Ошибок кода вручную на этом этапе я не исправлял. Затем я согласовал три аналитических вопроса; агент предложил SQL-фильтры, помог вынести запрос в функцию и подготовил график. Он обнаружил проблему кодировки при записи русского текста через PowerShell и исправил её записью файлов в UTF-8. Фильтры и выводы нужно понимать и уметь объяснить самостоятельно.

**Что будет, если запустить агента в папке с одним ноутбуком без `CLAUDE.md`:** агент сможет прочитать доступный ноутбук и инструкции пользователя, но не получит правила и описание проекта из отсутствующего файла. Поэтому придётся объяснять структуру, стиль работы и ограничения в запросе. Само отсутствие `CLAUDE.md` не означает отсутствие всех ограничений: у инструмента могут быть собственные правила и разрешения.

В `CLAUDE.md` добавлен раздел обо мне: я новичок, изучаю инфраструктуру ML и хочу сначала получать объяснение, затем код и способ проверки.


---

## Задание 11. README

В `README.md` добавлен раздел «Об этом репозитории»: автор — Малиновский Максимилиан, назначение учебного форка, структура файлов и последовательность запуска от Python 3.11 и `.venv` до выбора ядра и выполнения ноутбука. Поток пока не указан.


---

## Что было непонятно

Я начинающий. На этом этапе понадобилась помощь агента с различиями между PowerShell и Ubuntu, форком и клоном, а также с настройкой отдельного Python 3.11. Личные впечатления и фактически затраченное время заполню после самостоятельного повторения шагов.

## Задание 12. Проверка перед сдачей

- Рабочая ветка: `hw01`.
- `nbstripout --install` включён в рабочем репозитории Ubuntu: выводы будут очищаться при добавлении ноутбука в git.
- Полный запуск с новым ядром `ml-course` прошёл: 11 кодовых ячеек выполнены сверху вниз, ошибок нет; в выполненной копии сохранены версия сервера, таблицы и график.
- Для сдачи ноутбуком сохраняется отдельная выполненная копия с результатом подключения и графиком. В git отправляется исходный ноутбук без выводов.
- Поток и ники проверяющих пока не получены; добавление проверяющих и публикация PR ещё не выполнены.
